# FIG03: Expected vs. observed ratios, recalculated from participant peptide quant.

```
FIG03: Expected vs. observed ratios, recalculated from participant peptide quant.

Two panels, both computed from each participant's *-pepQuant peptide intensities:
  FIG03A  per-submission facets : observed log2(A/C) per peptide vs. abundance,
                                  dashed = design-expected log2(A/C) per species.
  FIG03B  recalculated summary  : A/B, B/C, A/C box-and-whisker by species
                                  (ratio of summed peptide intensity per species).

Self-contained; mirrors the curation in supp_reported_sample_ratios.ipynb.

DATA-TYPE HANDLING (verified from quantity magnitudes):
  raw    {01,02,03,04,07A,07B,09,11,12,14A,14B} : intensities  -> x=log2(B), y=log2(A/C)
  log2   {06}   : already log2                    -> x=B,        y=A-C
  scaled {10A,10B} : normalised to 300 %          -> y=log2(A/C) valid; x=log2(B) is a
                     percentage (abundance NOT comparable -> shaded facet)
  counts {41}   : spectral-count-like integers    -> x=log2(B+1), y=log2((A+1)/(C+1))
                     (ratio count-based -> flagged)

EXCLUDED: 05, 42, 14/Templates (placeholder 'FAKEPEPTIDE' data).
Multi-workflow labs kept as A/B: 07A/07B (120/90 min), 10A/10B (QE/Fusion+FAIMS),
14A/14B (GPF/Prosit library). Fixups: site-10-QEx mislabeled Sample C col; site-09
per-replicate columns averaged.
```

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

DATA_DIR = r"D:/2022 Multi-Species Standard Study"
OUTPUT = "output"
DATA = "data"
os.makedirs(OUTPUT, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

CURATED = {   # label -> pepQuant file (relative to DATA_DIR)
    "01": "01/01-pepQuant.tsv", "02": "02/02-pepQuant.csv", "03": "03/03-pepQuant.tsv",
    "04": "04/04-pepQuant.tsv", "06": "06/06-pepQuant.tsv",
    "07A": "07/07-pepQuant.tsv", "07B": "07/07-pepQuant_ShortGrad.tsv",
    "09": "09/09-pepQuant.tsv",
    "10A": "10/10-QEx_pepQuant.tsv", "10B": "10/10-Fusion_pepQuant.tsv",
    "11": "11/11-pepQuant.tsv", "12": "12/12-pepQuant.tsv",
    "14A": "14/14_pepQuant_sPRG_lumos_DIA_1x8mzStag_3x4mzGPFlibrary.txt",
    "14B": "14/14_pepQuant_sPRG_lumos_DIA_2x4mzStag_Prosit_library.txt",
    "41": "41/41-pepQuant.tsv",
}
ORDER = list(CURATED.keys())
DATATYPE = {k: "raw" for k in CURATED}
DATATYPE.update({"06": "log2", "10A": "scaled", "10B": "scaled", "41": "counts"})
NONCOMPARABLE_X = {"scaled", "counts"}

SPECIES_MAP = {"cow": "Bovine", "bovin": "Bovine", "bovine": "Bovine", "bos taurus": "Bovine",
               "human": "Human", "homo sapiens": "Human",
               "trout": "Trout", "salvelinus namaycush": "Trout", "salnm": "Trout"}
SPECIES_ORDER = ["Bovine", "Human", "Trout"]
COLORS = {"Bovine": "#E69F00", "Human": "#56B4E9", "Trout": "#009E73"}
COMPARISONS = ["A/B", "B/C", "A/C"]

MIX = {"A": {"Trout": 50, "Human": 45, "Bovine": 5},
       "B": {"Trout": 50, "Human": 20, "Bovine": 30},
       "C": {"Trout": 50, "Human": 3,  "Bovine": 47}}
EXPECTED_AC = {sp: np.log2(MIX["A"][sp] / MIX["C"][sp]) for sp in SPECIES_ORDER}   # facets
EXPECTED = {c: {sp: MIX[c[0]][sp] / MIX[c[-1]][sp] for sp in SPECIES_ORDER}        # box-whisker
            for c in COMPARISONS}

In [2]:
def _read(path):
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    sep = "," if path.lower().endswith(".csv") else "\t"
    return pd.read_csv(path, sep=sep, engine="python", on_bad_lines="skip")

In [3]:
def _load_quant(label, pq):
    """Return (df, species, qA, qB, qC) with per-file fixups applied."""
    df = _read(os.path.join(DATA_DIR, pq))
    df.columns = [str(c).strip() for c in df.columns]
    # site 10-QEx: 3rd quantity col mislabeled "Sample B Quantity" (dup) = Sample C
    if "Sample B Quantity.1" in df.columns and "Sample C Quantity" not in df.columns:
        df = df.rename(columns={"Sample B Quantity.1": "Sample C Quantity"})
    scol = "Species" if "Species" in df.columns else \
           ("PG.Organisms" if "PG.Organisms" in df.columns else None)
    if scol is None:
        raise ValueError(f"{label}: no species column ({list(df.columns)})")
    species = df[scol].map(lambda x: SPECIES_MAP.get(str(x).strip().lower()))
    q = {}
    for L in ("A", "B", "C"):
        cols = [c for c in df.columns if re.search(rf"Sample {L}(_R\d+)? Quantity", c)]
        if not cols:
            raise ValueError(f"{label}: no 'Sample {L} Quantity' column in {pq}")
        q[L] = df[cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
    return df, species, q["A"], q["B"], q["C"]

In [4]:
def load_peptides():
    """Per-peptide tidy frame: label, datatype, species, x_metric, y_metric=log2(A/C)."""
    frames = []
    for label, pq in CURATED.items():
        _, species, qA, qB, qC = _load_quant(label, pq)
        dt = DATATYPE[label]
        if dt in ("raw", "scaled"):
            keep = (qA > 0) & (qB > 0) & (qC > 0)
            x = np.log2(qB.where(keep)); y = np.log2((qA / qC).where(keep))
        elif dt == "log2":
            keep = qA.notna() & qB.notna() & qC.notna()
            x = qB.where(keep); y = (qA - qC).where(keep)
        elif dt == "counts":
            keep = qA.notna() & qB.notna() & qC.notna()
            x = np.log2(qB.where(keep) + 1); y = np.log2((qA.where(keep) + 1) / (qC.where(keep) + 1))
        sub = pd.DataFrame({"label": label, "datatype": dt, "species": species,
                            "x_metric": x, "y_metric": y})
        frames.append(sub[sub["species"].isin(SPECIES_ORDER)].dropna(subset=["x_metric", "y_metric"]))
    return pd.concat(frames, ignore_index=True)

In [5]:
def load_recalculated():
    """Per-submission ratio of summed peptide intensity per species -> A/B, B/C, A/C."""
    rows = []
    for label, pq in CURATED.items():
        df, species, qA, qB, qC = _load_quant(label, pq)
        g = pd.DataFrame({"species": species, "qA": qA, "qB": qB, "qC": qC}) \
            .dropna(subset=["species"]).groupby("species")[["qA", "qB", "qC"]].sum()
        for sp, r in g.iterrows():
            for comp, val in {"A/B": r.qA / r.qB, "B/C": r.qB / r.qC, "A/C": r.qA / r.qC}.items():
                if np.isfinite(val) and val > 0:
                    rows.append(dict(label=label, species=sp, comparison=comp, ratio=val))
    return pd.DataFrame(rows)

In [6]:
def fig03a_facets(p, out_png):
    n = len(ORDER); ncol, nrow = 4, int(np.ceil(n / 4))
    y_lo, y_hi = np.nanpercentile(p["y_metric"], [1, 99]); ypad = 0.08 * (y_hi - y_lo)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 2.8 * nrow), squeeze=False, sharey=True)
    for i, label in enumerate(ORDER):
        ax = axes[divmod(i, ncol)]; sub = p[p["label"] == label]; dt = DATATYPE[label]
        for sp in SPECIES_ORDER:
            ss = sub[sub["species"] == sp]
            if not ss.empty:
                ax.scatter(ss["x_metric"], ss["y_metric"], s=6, alpha=0.35, color=COLORS[sp], linewidth=0)
            ax.axhline(EXPECTED_AC[sp], ls="--", lw=1.0, color=COLORS[sp], zorder=3)
        tag = "" if dt == "raw" else f"  [{dt}]"
        ax.set_title(f"{label}{tag}", fontsize=10, color=("black" if dt == "raw" else "#B00000"))
        ax.set_ylim(y_lo - ypad, y_hi + ypad); ax.grid(True, lw=0.3)
        if dt in NONCOMPARABLE_X:
            ax.set_facecolor("#f7f2ea")
    for j in range(n, nrow * ncol):
        axes[divmod(j, ncol)].axis("off")
    handles = [Line2D([0], [0], marker="o", ls="", color=COLORS[s], label=s) for s in SPECIES_ORDER]
    handles += [Line2D([0], [0], ls="--", color="0.3", label="expected log2(A/C)")]
    axes[divmod(n, ncol)].legend(handles=handles, loc="center", frameon=False, fontsize=10)
    fig.supxlabel("abundance  (raw/scaled: log2 B | log2: B | counts: log2(B+1))   "
                  "— shaded = abundance not comparable")
    fig.supylabel("observed log2(A/C)")
    fig.suptitle("FIG03A  Expected vs. observed log2(A/C) per submission", fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    fig.savefig(out_png, dpi=200); fig.savefig(out_png.replace(".png", ".pdf")); plt.close(fig)
    print(f"  wrote {out_png}")

In [7]:
def fig03b_boxwhisker(p, out_png):
    fig, axes = plt.subplots(3, 1, figsize=(5.4, 7.8), sharex=True)
    for ax, comp in zip(axes, COMPARISONS):
        subc = p[p["comparison"] == comp]
        data = [subc.loc[subc["species"] == sp, "ratio"].values for sp in SPECIES_ORDER]
        ax.boxplot(data, orientation="horizontal", tick_labels=SPECIES_ORDER, widths=0.6, showfliers=False)
        ax.set_xscale("log")
        rng = np.random.default_rng(0)
        for i, sp in enumerate(SPECIES_ORDER, start=1):
            ss = subc[subc["species"] == sp]
            if not ss.empty:
                y = np.full(len(ss), i) + rng.uniform(-0.09, 0.09, size=len(ss))
                ax.scatter(ss["ratio"], y, s=18, alpha=0.8, color=COLORS[sp], edgecolor="black", linewidth=0.3, zorder=3)
            ax.axvline(EXPECTED[comp][sp], ls="--", lw=1.1, color=COLORS[sp], zorder=1)
        ax.set_title(comp, fontsize=11); ax.set_xlim(0.02, 30); ax.grid(True, axis="x", linewidth=0.3)
    axes[-1].set_xlabel("Ratio (log scale) — dashed = design-expected")
    fig.suptitle("FIG03B  Recalculated ratios from peptide quant (n=%d)" % p["label"].nunique(), fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(out_png, dpi=200); fig.savefig(out_png.replace(".png", ".pdf")); plt.close(fig)
    print(f"  wrote {out_png}")

In [8]:
# ---- generate figures ----
p = load_peptides()
rec = load_recalculated()
p.to_csv(os.path.join(DATA, "fig03_peptide_ratios_long.csv"), index=False)
rec.to_csv(os.path.join(DATA, "fig03_recalculated_ratios_long.csv"), index=False)
print(f"{len(p)} peptide points, {p['label'].nunique()} submissions")
fig03a_facets(p, os.path.join(OUTPUT, "FIG03A_expected_vs_observed_facets.png"))
fig03b_boxwhisker(rec, os.path.join(OUTPUT, "FIG03B_recalculated_ratios.png"))

137056 peptide points, 15 submissions


  wrote output\FIG03A_expected_vs_observed_facets.png


  wrote output\FIG03B_recalculated_ratios.png
